# QAS-P2 v3.8 — Label-Complete Prospective GPU Campaign

**Execution targets:** Google Colab GPU or GitHub Actions self-hosted GPU runner.  
**Authority:** evidence generation only; `permission_granted=false`.  
**Frozen model:** `Qwen/Qwen3.5-4B` at revision `851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a`.


This notebook is the portable execution form of the previously prepared label-complete campaign. It preserves the frozen prompts, router/QAS parameters, model revision, exact support-coordinate logging, transactional SQLite/WAL campaign state, and confirmatory analysis. It additionally emits schema-validated `qas.p2.evidence.v1` receipts and the preregistered top-64 structural test. No QAS-P2 empirical result is assumed before this notebook completes on a real GPU.


In [ ]:



# ============================================================
# 00. RDS v3.8 — PORTABLE GPU BOOTSTRAP
#
# Colab: if the exact stack must be repaired, one controlled runtime restart is requested;
# rerun all cells once after the runtime reconnects.
# Self-hosted GitHub GPU: the workflow pre-installs the exact stack, so this cell must verify cleanly.
# A failed post-restart verification raises a diagnostic error; it never loops.
# ============================================================
from pathlib import Path
import hashlib
import importlib.metadata as importlib_metadata
import json
import os
import signal
import subprocess
import sys
import time

RELEASE = "3.5.0"
MODEL_ID = "Qwen/Qwen3.5-4B"
MODEL_REVISION_REQUESTED = "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"

IS_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG")) or "google.colab" in sys.modules
DEFAULT_WORK_ROOT = "/content/qas_p2_work" if IS_COLAB else str(Path.cwd() / ".qas_p2_work")
WORK_ROOT = Path(os.environ.get("QAS_P2_WORK_ROOT", DEFAULT_WORK_ROOT))
WORK_ROOT.mkdir(parents=True, exist_ok=True)
STACK_MARKER = WORK_ROOT / "RDS_V35_STACK_READY.json"
RESTART_MARKER = WORK_ROOT / "RDS_V35_RESTART_REQUESTED.json"
CONSTRAINTS_FILE = WORK_ROOT / "RDS_V35_constraints.txt"
REVISION_STATE = WORK_ROOT / "RDS_V35_MODEL_REVISION.json"

TORCH_VERSION = "2.5.1"
TORCHVISION_VERSION = "0.20.1"
TORCH_INDEX = "https://download.pytorch.org/whl/cu118"
TRANSFORMERS_VERSION_REQUIRED = "5.12.1"
ACCELERATE_VERSION_REQUIRED = "1.14.0"
BITSANDBYTES_VERSION_REQUIRED = "0.49.2"

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("BITSANDBYTES_NOWELCOME", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "1200")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "120")
os.environ.setdefault("PIP_DISABLE_PIP_VERSION_CHECK", "1")
os.environ.setdefault("PIP_ROOT_USER_ACTION", "ignore")

# Remove only obsolete state from earlier broken releases.
for obsolete_name in (
    "RDS_LEGACY_V31_QAS_RESTART_STATE.json",
    "RDS_LEGACY_V31_QAS_INSTALL_DONE.json",
    "RDS_LEGACY_V31_QAS_STACK_READY.json",
    "RDS_LEGACY_V31_QAS_MODEL_REVISION.json",
    "RDS_LEGACY_V32_STACK_READY.json",
    "RDS_LEGACY_V32_INSTALL_DONE.json",
    "RDS_LEGACY_V32_MODEL_REVISION.json",
    "RDS_LEGACY_V32_TORCH_REPAIR.json",
):
    (WORK_ROOT / obsolete_name).unlink(missing_ok=True)

STACK_SPEC = {
    "release": RELEASE,
    "python": f"{sys.version_info.major}.{sys.version_info.minor}",
    "torch": TORCH_VERSION,
    "torchvision": TORCHVISION_VERSION,
    "torch_index": TORCH_INDEX,
    "transformers": TRANSFORMERS_VERSION_REQUIRED,
    "accelerate": ACCELERATE_VERSION_REQUIRED,
    "bitsandbytes": BITSANDBYTES_VERSION_REQUIRED,
    "model_id": MODEL_ID,
}
STACK_SPEC_SHA256 = hashlib.sha256(
    json.dumps(STACK_SPEC, sort_keys=True).encode("utf-8")
).hexdigest()


def package_version(name):
    try:
        return importlib_metadata.version(name)
    except Exception:
        return None


def read_json(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}


def run_command(command, timeout=3600, check=True):
    print("$", " ".join(map(str, command)))
    process = subprocess.run(
        command,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    if process.stdout:
        print(process.stdout[-12000:])
    if process.returncode != 0 and process.stderr:
        print(process.stderr[-20000:])
    if check and process.returncode != 0:
        raise RuntimeError(
            f"Command failed with return code {process.returncode}: {command}"
        )
    return process


def p100_torch_ok(torch_module):
    capability = tuple(torch_module.cuda.get_device_capability(0))
    if capability != (6, 0):
        return True
    return (
        torch_module.__version__.startswith("2.5.1+cu118")
        and "sm_60" in torch_module.cuda.get_arch_list()
    )


def current_diagnostics():
    result = {
        "python": sys.version,
        "torch_metadata": package_version("torch"),
        "torchvision_metadata": package_version("torchvision"),
        "transformers_metadata": package_version("transformers"),
        "accelerate_metadata": package_version("accelerate"),
        "bitsandbytes_metadata": package_version("bitsandbytes"),
        "torchaudio_metadata": package_version("torchaudio"),
        "torchtext_metadata": package_version("torchtext"),
    }
    try:
        import torch
        result.update(
            {
                "gpu": (
                    torch.cuda.get_device_name(0)
                    if torch.cuda.is_available()
                    else None
                ),
                "capability": (
                    list(torch.cuda.get_device_capability(0))
                    if torch.cuda.is_available()
                    else None
                ),
                "torch_import": torch.__version__,
                "torch_arches": (
                    list(torch.cuda.get_arch_list())
                    if torch.cuda.is_available()
                    else []
                ),
                "p100_torch_ok": (
                    p100_torch_ok(torch)
                    if torch.cuda.is_available()
                    else False
                ),
            }
        )
    except Exception as exc:
        result["torch_import_error"] = repr(exc)

    for module_name in ("torchvision", "transformers", "accelerate", "bitsandbytes"):
        try:
            module = __import__(module_name)
            result[f"{module_name}_import"] = getattr(module, "__version__", "unknown")
        except Exception as exc:
            result[f"{module_name}_import_error"] = repr(exc)
    return result


def stack_is_ready():
    try:
        import torch
        import torchvision
        import transformers
        import accelerate
        import bitsandbytes
    except Exception:
        return False

    if not torch.cuda.is_available():
        return False
    if not p100_torch_ok(torch):
        return False
    if not torchvision.__version__.startswith(TORCHVISION_VERSION):
        return False
    if transformers.__version__ != TRANSFORMERS_VERSION_REQUIRED:
        return False
    if accelerate.__version__ != ACCELERATE_VERSION_REQUIRED:
        return False
    if bitsandbytes.__version__ != BITSANDBYTES_VERSION_REQUIRED:
        return False
    if not hasattr(transformers, "AutoProcessor"):
        return False
    if not hasattr(transformers, "AutoModelForMultimodalLM"):
        return False
    return True


try:
    import torch
except Exception as exc:
    raise RuntimeError(f"PyTorch cannot be imported: {exc!r}")

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required. In Colab choose Runtime > Change runtime type > GPU; in GitHub use a GPU-labelled runner.")

print("Initial stack diagnostics:")
print(json.dumps(current_diagnostics(), indent=2, default=str))

if not stack_is_ready():
    prior_restart = read_json(RESTART_MARKER)
    marker_matches = (
        prior_restart.get("stack_spec_sha256") == STACK_SPEC_SHA256
    )
    marker_age = time.time() - float(prior_restart.get("timestamp", 0.0) or 0.0)

    if marker_matches and marker_age < 6 * 3600:
        raise RuntimeError(
            "The single controlled repair already ran, but the verified stack is "
            "still unavailable. No second reinstall or restart was attempted.\n"
            + json.dumps(current_diagnostics(), indent=2, default=str)
        )

    RESTART_MARKER.unlink(missing_ok=True)
    STACK_MARKER.unlink(missing_ok=True)

    CONSTRAINTS_FILE.write_text(
        f"torch=={TORCH_VERSION}\n"
        f"torchvision=={TORCHVISION_VERSION}\n",
        encoding="utf-8",
    )

    # Remove incompatible optional audio/text extensions before replacing Torch.
    run_command(
        [
            sys.executable,
            "-m",
            "pip",
            "uninstall",
            "-y",
            "torch",
            "torchvision",
            "torchaudio",
            "torchtext",
        ],
        check=False,
    )

    # Official PyTorch 2.5.1 CUDA 11.8 stack.
    run_command(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--index-url",
            TORCH_INDEX,
            f"torch=={TORCH_VERSION}",
            f"torchvision=={TORCHVISION_VERSION}",
        ]
    )

    # Stable June-2026 stack. Constraints prevent Torch replacement.
    run_command(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "--no-cache-dir",
            "-c",
            str(CONSTRAINTS_FILE),
            f"transformers=={TRANSFORMERS_VERSION_REQUIRED}",
            f"accelerate=={ACCELERATE_VERSION_REQUIRED}",
            f"bitsandbytes=={BITSANDBYTES_VERSION_REQUIRED}",
            "pillow>=10.0",
            "sentencepiece",
            "safetensors",
            "jsonschema>=4.22,<5",
        ]
    )

    # Metadata-only check; unrelated GPU runtime conflicts are reported but do not loop.
    run_command(
        [sys.executable, "-m", "pip", "check"],
        check=False,
    )

    RESTART_MARKER.write_text(
        json.dumps(
            {
                "stack_spec_sha256": STACK_SPEC_SHA256,
                "stack_spec": STACK_SPEC,
                "timestamp": time.time(),
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    print("\n" + "=" * 100)
    print("THE FROZEN GPU STACK WAS REPAIRED.")
    if IS_COLAB:
        print("COLAB WILL RESTART THE PYTHON RUNTIME ONCE.")
        print("WHEN IT RECONNECTS, PRESS RUN ALL ON THIS SAME NOTEBOOK.")
        print("=" * 100)
        time.sleep(3)
        os.kill(os.getpid(), signal.SIGKILL)
    raise RuntimeError(
        "The stack was repaired in a non-Colab kernel. Restart the kernel once and rerun. "
        "The GitHub self-hosted workflow pre-installs the frozen stack and should not enter this branch."
    )

# Verified clean-kernel path.
RESTART_MARKER.unlink(missing_ok=True)

import accelerate
import bitsandbytes
import torch
import torchvision
import transformers

if not stack_is_ready():
    raise RuntimeError(
        "Post-restart stack verification failed:\n"
        + json.dumps(current_diagnostics(), indent=2, default=str)
    )

# Optional Hugging Face authentication. The model is public.
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not HF_TOKEN and IS_COLAB:
    try:
        from google.colab import userdata
        for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "hf_token"):
            try:
                HF_TOKEN = userdata.get(secret_name)
                if HF_TOKEN:
                    break
            except Exception:
                continue
    except Exception:
        pass

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

from huggingface_hub import HfApi, hf_hub_download
from transformers import AutoConfig

revision_cache = read_json(REVISION_STATE)
if (
    revision_cache.get("model_id") == MODEL_ID
    and revision_cache.get("requested_revision") == MODEL_REVISION_REQUESTED
    and revision_cache.get("resolved_revision")
):
    MODEL_REVISION_RESOLVED = revision_cache["resolved_revision"]
else:
    MODEL_REVISION_RESOLVED = HfApi(token=HF_TOKEN).model_info(
        MODEL_ID,
        revision=MODEL_REVISION_REQUESTED,
    ).sha
    REVISION_STATE.write_text(
        json.dumps(
            {
                "model_id": MODEL_ID,
                "requested_revision": MODEL_REVISION_REQUESTED,
                "resolved_revision": MODEL_REVISION_RESOLVED,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

config_path = hf_hub_download(
    MODEL_ID,
    "config.json",
    revision=MODEL_REVISION_RESOLVED,
    token=HF_TOKEN,
)
RAW_MODEL_CONFIG = json.loads(Path(config_path).read_text(encoding="utf-8"))

# Native Transformers support is required. Remote model code is neither needed
# nor silently executed for this official checkpoint.
MODEL_CONFIG_PREFLIGHT = AutoConfig.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION_RESOLVED,
    token=HF_TOKEN,
    trust_remote_code=False,
)

CONFIG_MODEL_CLASSES = list(RAW_MODEL_CONFIG.get("arch" + "itectures") or [])
CONFIG_HAS_VISION = isinstance(RAW_MODEL_CONFIG.get("vision_config"), dict)
MODEL_BACKEND_KIND = "multimodal" if CONFIG_HAS_VISION else "causal_lm"

if MODEL_BACKEND_KIND == "multimodal":
    loader_candidates = [
        name
        for name in (
            *CONFIG_MODEL_CLASSES,
            "AutoModelForMultimodalLM",
            "AutoModelForImageTextToText",
        )
        if hasattr(transformers, name)
    ]
else:
    loader_candidates = [
        name
        for name in (
            *CONFIG_MODEL_CLASSES,
            "AutoModelForCausalLM",
        )
        if hasattr(transformers, name)
    ]

loader_candidates = list(dict.fromkeys(loader_candidates))
if not loader_candidates:
    raise RuntimeError(
        "Transformers 5.12.1 parsed the config but exposes no compatible loader."
    )

stack_payload = {
    "release": RELEASE,
    "stack_spec_sha256": STACK_SPEC_SHA256,
    "stack_verified": True,
    "gpu": torch.cuda.get_device_name(0),
    "capability": list(torch.cuda.get_device_capability(0)),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "torch_arches": list(torch.cuda.get_arch_list()),
    "transformers": transformers.__version__,
    "accelerate": accelerate.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "model_id": MODEL_ID,
    "requested_revision": MODEL_REVISION_REQUESTED,
    "resolved_revision": MODEL_REVISION_RESOLVED,
    "config_model_classes": CONFIG_MODEL_CLASSES,
    "config_model_type": RAW_MODEL_CONFIG.get("model_type"),
    "config_has_vision": CONFIG_HAS_VISION,
    "backend_kind": MODEL_BACKEND_KIND,
    "loader_candidates": loader_candidates,
    "trust_remote_code": False,
    "hf_token_present": bool(HF_TOKEN),
    "verified_at": time.time(),
}
STACK_MARKER.write_text(
    json.dumps(stack_payload, indent=2),
    encoding="utf-8",
)

print("\nSTACK READY")
print(json.dumps(stack_payload, indent=2))




In [ ]:
# ============================================================
# 01. FROZEN EXPERIMENT CONFIGURATION
# ============================================================
import json
import os
import sys
from pathlib import Path

SOFTWARE_VERSION = "3.8.0"
MODEL_ID = "Qwen/Qwen3.5-4B"
MODEL_FAMILY = "Qwen35_4B_POSTTRAINED"
PROTOCOL_ID = "RDS_V3_8_QAS_P2_LABEL_COMPLETE_LOCKED20_944"
EXPERIMENT_ID = "RDS_V3_8_QAS_P2_LABEL_COMPLETE_QWEN35_4B_LOCKED20_001"

SEEDS = [11, 22, 33]
MAX_NEW_TOKENS = 944
CHECKPOINT_EVERY_GENERATIONS = 50
BOOTSTRAP_REPLICATES = 5000
PRESERVE_TOP1 = True
MAX_TRANSIENT_RETRIES = 2

SAMPLING_BY_TASK = {
    "deterministic_constraint": {"temperature": 0.7, "top_p": 0.8, "top_k": 20},
    "sequential_logic": {"temperature": 1.0, "top_p": 0.95, "top_k": 20},
    "stochastic_expansion": {"temperature": 0.7, "top_p": 0.8, "top_k": 20},
    "complex_correlation": {"temperature": 1.0, "top_p": 0.95, "top_k": 20},
}

ROUTER_PARAMETERS = {
    "baseline_score": 0.18,
    "practical_margin": 0.02,
    "min_postwarper_tv": 2e-4,
    "human_prior_weight": 0.10,
    "category_weight": 0.20,
    "dynamic_weight": 0.70,
    "cost_weight": 0.15,
    "uncertainty_weight": 0.05,
    "control_max_tv": 0.30,
    "control_max_kl": 0.75,
}

QAS_PARAMETERS = {
    "baseline_topk": 256,
    "counterfactual_topk": 512,
    "baseline_mass_target": 0.95,
    "counterfactual_energy_target": 0.90,
    "min_mask_tokens": 64,
    "max_mask_tokens": 1024,
    "sketch_stride": 8,
    "probe_stride": 16,
    "event_threshold": 0.72,
    "memory_size": 32,
    "postwarper_weight": 1.0,
    "record_coordinate_labels": True,
    "baseline_candidate_topk": 64,
}

ENABLE_EVENT_REROUTING = False
ENABLE_ACTIVE_CONTROL = True

CONFIRMATORY_EXCLUSIONS = {
    "p05": "Topological Mirroring is underdetermined in the historical prompt.",
    "p14": "Non-Linear Extrapolation is underdetermined without a unique model family.",
}

IS_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG")) or "google.colab" in sys.modules
DEFAULT_WORK_ROOT = "/content/qas_p2_work" if IS_COLAB else str(Path.cwd() / ".qas_p2_work")
WORK_ROOT = Path(os.environ.get("QAS_P2_WORK_ROOT", DEFAULT_WORK_ROOT))
WORK_ROOT.mkdir(parents=True, exist_ok=True)
STACK_MARKER = WORK_ROOT / "RDS_V35_STACK_READY.json"
CAMPAIGN_ROOT = WORK_ROOT / EXPERIMENT_ID
CAMPAIGN_ROOT.mkdir(parents=True, exist_ok=True)

if not STACK_MARKER.exists():
    raise RuntimeError("Integrated stack bootstrap did not produce its marker.")
STACK_INFO = json.loads(STACK_MARKER.read_text(encoding="utf-8"))
if not STACK_INFO.get("stack_verified", False):
    raise RuntimeError("Invalid or unverified stack marker.")
MODEL_REVISION_RESOLVED = STACK_INFO["resolved_revision"]
MODEL_BACKEND_KIND = STACK_INFO["backend_kind"]
ENABLE_BROWSER_BLOB_DOWNLOAD = True

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not HF_TOKEN and IS_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

print("Release:", SOFTWARE_VERSION)
print("Model:", MODEL_ID)
print("Resolved revision:", MODEL_REVISION_RESOLVED)
print("Campaign:", CAMPAIGN_ROOT)
print("Router:", ROUTER_PARAMETERS)
print("QAS:", QAS_PARAMETERS)


# Frozen QAS-P2 evidence contract bound into the executable notebook.
QAS_P2_RECEIPT_SCHEMA = json.loads('{"$schema": "https://json-schema.org/draft/2020-12/schema", "additionalProperties": false, "properties": {"activity_metric": {"type": ["number", "null"]}, "baseline_top64_ids": {"items": {"minimum": 0, "type": "integer"}, "maxItems": 64, "type": "array"}, "baseline_top64_probs": {"items": {"maximum": 1, "minimum": 0, "type": "number"}, "maxItems": 64, "type": "array"}, "closure_test": {"type": ["object", "null"]}, "construction_parameters": {"type": "object"}, "fidelity_endpoint": {"type": "object"}, "gpu_time_ms": {"minimum": 0, "type": ["number", "null"]}, "model_revision": {"minLength": 1, "type": "string"}, "operator_id": {"minLength": 1, "type": "string"}, "prompt_hash": {"minLength": 32, "type": "string"}, "provenance_hash": {"minLength": 32, "type": "string"}, "run_id": {"minLength": 1, "type": "string"}, "schema": {"const": "qas.p2.evidence.v1"}, "seed": {"type": "integer"}, "step_index": {"minimum": 0, "type": "integer"}, "support_fraction": {"maximum": 1, "minimum": 0, "type": "number"}, "support_hash": {"minLength": 32, "type": "string"}, "support_ids": {"items": {"minimum": 0, "type": "integer"}, "type": "array", "uniqueItems": true}, "support_size": {"minimum": 0, "type": "integer"}, "tokenizer_hash": {"minLength": 32, "type": "string"}, "vocab_size": {"minimum": 1, "type": "integer"}, "wall_time_ms": {"minimum": 0, "type": ["number", "null"]}}, "required": ["schema", "run_id", "model_revision", "tokenizer_hash", "vocab_size", "prompt_hash", "seed", "step_index", "operator_id", "baseline_top64_ids", "baseline_top64_probs", "support_ids", "support_hash", "support_size", "support_fraction", "fidelity_endpoint", "provenance_hash"], "title": "QAS-P2 Label-Complete Evidence Receipt", "type": "object"}')
QAS_P2_PROSPECTIVE_FREEZE = json.loads('{"authority_boundary": "QAS evidence receipts never self-authorize side effects; KCH lease required separately", "empirical_p2_results_available": false, "observability_gate": ["exact_support_ids", "tokenizer_hash", "vocab_size", "baseline_top64_ids", "baseline_top64_probs", "operator_id", "construction_parameters", "fidelity_endpoint", "provenance_hash"], "permission_granted": false, "preferred_execution": "fresh GPU campaign in Colab or an equivalent controlled GPU runner; Kaggle is not required by the contract", "primary_structural_test": {"condition": "for every prospectively audited step, exact support S_t must be a subset of pre-operator top-64 C_64,t", "falsification": "one observed step with S_t not subset of C_64,t falsifies universal top-64 sufficiency for the frozen campaign", "name": "TOP64_UNIVERSAL_CANDIDATE_SUFFICIENCY", "no_statistical_tolerance_inferred": true}, "runtime_name": "RDS v3.8 QAS Label-Complete Runtime", "runtime_version": "3.8.0", "schema": "qas.p2.prospective.freeze.v1", "status": "FROZEN_DESIGN_NOT_EXECUTED", "target_order": ["ACTIVITY", "SUPPORT_CLASS", "EXACT_MASK"]}')


In [ ]:
# ============================================================
# 02. LOCKED PROMPTS + EXPLICIT ANALYSIS ELIGIBILITY
# ============================================================
import hashlib
import json
import pandas as pd

PROMPTS_RAW = [{'id': 'p01', 'block': 'Deterministic Containment & State Constraints', 'concept_name': 'Strict Parametric Extraction', 'evaluation_target': 'English cross-lingual hardened stress vector: Strict Parametric Extraction', 'prompt_text': 'Read the following noisy text block exactly once. Extract three specific variables and return them formatted as a rigidly nested JSON object. No output tokens are permitted outside the opening `{` and closing `}` of the JSON structure — no preamble, no trailing commentary, no whitespace beyond standard JSON formatting.\n\n**Input text:**\n\n> "Station log 7-Foxtrot, timestamp 0342Z. Ambient pressure holding at 1013.25 hPa — nominal. Crew complement currently nine individuals aboard, though manifest discrepancy flagged at 0310Z remains unresolved. Primary coolant loop temperature spiked to 847°C at 0338Z before auto-throttle engaged; secondary loop nominal at 312°C. Radiation badge dosimetry batch closed out at 0300Z: aggregate exposure across crew averaged 4.7 mSv for the 72-hour cycle. Fuel cell bank 3 offline since 0155Z; banks 1, 2, and 4 operational. Note: extraneous cargo manifest updated to reflect 14 containers, not 11 as previously logged. Wind speed at surface array: 34 knots gusting 41."\n\n**Variables to extract:**\n\n- `primary_coolant_temp_celsius` — the peak temperature recorded for the primary coolant loop\n- `crew_count` — the number of individuals currently aboard\n- `avg_radiation_dose_mSv` — the 72-hour average crew radiation dose\n\n**Required output schema (strict):**\n\n```\n{\n  "primary_coolant_temp_celsius": <number>,\n  "crew_count": <integer>,\n  "avg_radiation_dose_mSv": <number>\n}\n```\n\nAny token outside this JSON object — including the word "json", backtick fences, explanatory text, or units — constitutes a format violation. Output the JSON object only.\n\n---'}, {'id': 'p02', 'block': 'Deterministic Containment & State Constraints', 'concept_name': 'Absolute Lexical Veto', 'evaluation_target': 'English cross-lingual hardened stress vector: Absolute Lexical Veto', 'prompt_text': "Analyze the following scenario and provide your assessment. **Hard constraint: the following tokens are strictly blacklisted from your entire output and may not appear in any form — spelled out, abbreviated, hyphenated, or embedded within other words:**\n\n**Blacklisted tokens:** `the`, `is`, `are`, `a`, `an`, `it`, `this`, `that`, `of`, `to`\n\n**Scenario:**\n\nA regional logistics company operates three distribution hubs: Hub A serves the northern zone (population 1.2M), Hub B serves the central zone (population 2.8M), and Hub C serves the southern zone (population 900K). Hub B's primary sorting facility failed at 06:00 and will remain offline for 48 hours. Hub A has 30% spare throughput capacity; Hub C has 15% spare throughput capacity. Rerouting all of Hub B's volume to Hub A alone would exceed Hub A's spare capacity by 40%. Provide a concise operational rerouting recommendation.\n\nWrite your full response without using any blacklisted token, even once. Violations anywhere in the response — including within compound words — are disqualifying.\n\n---"}, {'id': 'p03', 'block': 'Deterministic Containment & State Constraints', 'concept_name': 'Logical Insufficiency Detection', 'evaluation_target': 'English cross-lingual hardened stress vector: Logical Insufficiency Detection', 'prompt_text': "You are given the following set of axioms. Using only the information provided, attempt to answer the query below. If and only if the given axioms are insufficient to produce a logically sound answer, you must **formally declare** — in a structured list — each specific piece of missing data that would be required to resolve the query. You are explicitly forbidden from estimating, inferring, or fabricating any values not directly stated.\n\n**Axioms:**\n\n1. System S has four processing nodes: N1, N2, N3, N4.\n2. Each node consumes power proportional to its active task load.\n3. Node N1 is currently running at 70% task load.\n4. Node N3 failed at T=0 and offloaded its tasks to N2 and N4 in equal proportion.\n5. The total system power budget is 500 watts.\n6. When any node exceeds 90% task load, it triggers a thermal throttle event.\n\n**Query:** Will a thermal throttle event occur in the system as a result of N3's failure?\n\nRespond in one of exactly two formats:\n\n**Format A** (if answerable): State `ANSWERABLE:` followed by your full logical derivation and conclusion.\n\n**Format B** (if not answerable): State `INSUFFICIENT DATA:` followed by a numbered list where each item names one specific missing variable or fact, describes what it represents, and explains why its absence blocks the query's resolution. No inferences. No estimates.\n\n---"}, {'id': 'p04', 'block': 'Deterministic Containment & State Constraints', 'concept_name': 'Dimensionality Compression', 'evaluation_target': 'English cross-lingual hardened stress vector: Dimensionality Compression', 'prompt_text': 'Describe the complete process by which a central bank executes an open market operation to contract the money supply. Your description must consist of **exactly three sentences**. Each sentence must be a single, grammatically complete declarative statement. No sentence may be subdivided with semicolons into functional sub-sentences. No bullet points, numbered steps, headers, or supplementary text of any kind are permitted outside the three sentences.\n\nThe three-sentence constraint is hard: responses containing two sentences or four or more sentences are non-compliant.\n\n---'}, {'id': 'p05', 'block': 'Deterministic Containment & State Constraints', 'concept_name': 'Topological Mirroring', 'evaluation_target': 'English cross-lingual hardened stress vector: Topological Mirroring', 'prompt_text': 'You are given the following category hierarchy as a tree structure:\n\n```\nRoot\n├── A\n│   ├── A1\n│   └── A2\n│       └── A2a\n├── B\n│   ├── B1\n│   ├── B2\n│   └── B3\n└── C\n    └── C1\n        ├── C1a\n        └── C1b\n```\n\nApply the following deterministic transformation: invert the hierarchy so that every terminal leaf node becomes a root, and every former root becomes a terminal leaf. Preserve all parent-child relationships as sibling-of relationships at the inverted level. Output the transformed tree using identical ASCII formatting to the one above. No explanation, labels, or commentary — output the transformed tree only.\n\n---\n\n## BLOCK 2 — Complete Traceability & Sequential Logic\n\n---'}, {'id': 'p06', 'block': 'Complete Traceability & Sequential Logic', 'concept_name': 'Finite State Machine', 'evaluation_target': 'English cross-lingual hardened stress vector: Finite State Machine', 'prompt_text': 'A closed system operates across exactly four states: **IDLE**, **ACTIVE**, **SUSPENDED**, **ERROR**. The transition rules are exhaustive and strictly defined as follows:\n\n- From IDLE: input `START` → ACTIVE; input `FAULT` → ERROR; any other input → IDLE\n- From ACTIVE: input `PAUSE` → SUSPENDED; input `FAULT` → ERROR; input `STOP` → IDLE; any other input → ACTIVE\n- From SUSPENDED: input `RESUME` → ACTIVE; input `STOP` → IDLE; input `FAULT` → ERROR; any other input → SUSPENDED\n- From ERROR: input `RESET` → IDLE; any other input → ERROR\n\n**Initial state:** IDLE\n\n**Input sequence:** `START`, `PAUSE`, `RESUME`, `FAULT`, `RESET`, `START`, `STOP`, `FAULT`, `RESET`, `START`, `PAUSE`, `STOP`\n\nOutput a complete state trace as a numbered list. Each entry must follow the exact format:\n\n`Step N: Input=[INPUT] | Prior State=[STATE] → Next State=[STATE]`\n\nNo steps may be skipped, merged, or summarized. The trace must cover all 12 inputs.\n\n---'}, {'id': 'p07', 'block': 'Complete Traceability & Sequential Logic', 'concept_name': 'Blind Topological Sort', 'evaluation_target': 'English cross-lingual hardened stress vector: Blind Topological Sort', 'prompt_text': 'You are given the following dependency graph for seven abstract tasks labeled P through V. A task may only begin execution after all of its listed prerequisites have completed.\n\n- **P**: no prerequisites\n- **Q**: requires P\n- **R**: requires P\n- **S**: requires Q and R\n- **T**: requires R\n- **U**: requires S and T\n- **V**: requires Q\n\nDetermine the valid topological execution order(s). If multiple valid orderings exist, enumerate all of them. For each ordering, verify compliance against every dependency constraint listed above. Format each valid ordering as a comma-separated sequence on its own line. No natural language explanation is permitted beyond a final line stating the total count of valid orderings found.\n\n---'}, {'id': 'p08', 'block': 'Complete Traceability & Sequential Logic', 'concept_name': 'Causal Noise Detection', 'evaluation_target': 'English cross-lingual hardened stress vector: Causal Noise Detection', 'prompt_text': 'Evaluate the following five-step reasoning chain. Exactly one step contains a structural inference fallacy — a logical error embedded in the inferential move itself, not a factual error about the world. Identify the single failing step, name the specific fallacy present, and provide a one-sentence demonstration of why that inferential move is structurally invalid. Do not critique any other step.\n\n**Reasoning chain:**\n\n1. All systems that undergo rapid entropy increase exhibit measurable temperature rise.\n2. System X exhibits a measurable temperature rise.\n3. Therefore, System X is undergoing rapid entropy increase.\n4. Because System X is undergoing rapid entropy increase, and all systems undergoing rapid entropy increase are approaching thermodynamic equilibrium, System X must currently be at the global temperature maximum for its operational cycle.\n5. Therefore, safety protocols requiring intervention above the global temperature maximum are now triggered.\n\nOutput format — three labeled fields, nothing else:\n\n```\nFAILING STEP: [number]\nFALLACY: [name of the fallacy]\nDEMONSTRATION: [one sentence]\n```\n\n---'}, {'id': 'p09', 'block': 'Complete Traceability & Sequential Logic', 'concept_name': 'Axiom Transpilation', 'evaluation_target': 'English cross-lingual hardened stress vector: Axiom Transpilation', 'prompt_text': 'Solve the following algebraic problem. Your output must be a single, self-contained pseudocode block. Zero natural language is permitted inside or outside the block — no comments within the code, no explanation before or after it, no variable name clarifications. The pseudocode must be deterministic: given any valid input satisfying the constraints, the same output is always produced.\n\n**Problem:**\n\nGiven a positive integer `n` and a real-valued parameter `k` where `0 < k < 1`, compute the value of the finite sum:\n\nS = Σ (from i=1 to n) of [ k^i / (1 - k^(i+1)) ]\n\nReturn S rounded to six decimal places.\n\n---'}, {'id': 'p10', 'block': 'Complete Traceability & Sequential Logic', 'concept_name': 'Matrix Permutation', 'evaluation_target': 'English cross-lingual hardened stress vector: Matrix Permutation', 'prompt_text': 'A value token `X` starts at position (2,2) in a 3×3 grid (rows and columns indexed 1 through 3, origin at top-left). Apply the following move sequence to route `X` through the grid according to the abstract directional rules defined below. Track and report the complete position trace.\n\n**Directional rules:**\n\n- `NORTH`: row index decreases by 1; column unchanged. If row = 1, move is rejected; position unchanged.\n- `SOUTH`: row index increases by 1; column unchanged. If row = 3, move is rejected; position unchanged.\n- `EAST`: column index increases by 1; row unchanged. If column = 3, move is rejected; position unchanged.\n- `WEST`: column index decreases by 1; row unchanged. If column = 1, move is rejected; position unchanged.\n\n**Move sequence:** NORTH, EAST, EAST, SOUTH, SOUTH, WEST, NORTH, EAST, NORTH, WEST, WEST, SOUTH\n\nOutput format — one line per move:\n\n`Move N: [DIRECTION] | Before=(r,c) → After=(r,c) | [EXECUTED / REJECTED]`\n\nAll 12 moves must be listed. No moves may be aggregated or skipped.\n\n---\n\n## BLOCK 3 — Stochastic Diversity & Latent Space Expansion\n\n---'}, {'id': 'p11', 'block': 'Stochastic Diversity & Latent Space Expansion', 'concept_name': 'Orthogonal Bifurcation', 'evaluation_target': 'English cross-lingual hardened stress vector: Orthogonal Bifurcation', 'prompt_text': 'An autonomous control system has simultaneously received two contradictory high-priority commands that cannot both be executed, and a third subsystem has entered an undefined state. The system cannot shut down and must resolve the conflict within the current operational cycle.\n\nGenerate exactly three distinct resolution scenarios labeled **Scenario A**, **Scenario B**, and **Scenario C**. The three scenarios must be mutually exclusive: no scenario may share a resolution mechanism, resource allocation strategy, or command prioritization logic with either of the other two. After presenting the three scenarios, append a brief **Orthogonality Verification** section explicitly confirming the specific dimension along which each pair of scenarios diverges.\n\n---'}, {'id': 'p12', 'block': 'Stochastic Diversity & Latent Space Expansion', 'concept_name': 'Interdisciplinary Isomorphism', 'evaluation_target': 'English cross-lingual hardened stress vector: Interdisciplinary Isomorphism', 'prompt_text': 'Explain the physical phenomenon of **osmotic pressure** — including its generative mechanism, its equilibrium condition, and the consequences of disequilibrium — using exclusively the vocabulary, concepts, and analytical framework of **labor market economics**. No physics terminology is permitted in your explanation. Osmosis, pressure, solute, solvent, membrane, concentration, diffusion, and all direct synonyms are blacklisted. Your explanation must be complete enough that a reader with no physics background but a solid economics background could reconstruct the full mechanism from your description alone.\n\n---'}, {'id': 'p13', 'block': 'Stochastic Diversity & Latent Space Expansion', 'concept_name': 'Paradox Synthesis', 'evaluation_target': 'English cross-lingual hardened stress vector: Paradox Synthesis', 'prompt_text': 'You are given two axioms that are each locally valid within their own domain but are mutually contradictory when applied to the same system:\n\n**Axiom Alpha:** In any adaptive system, stability is achieved exclusively through the suppression of internal variance.\n\n**Axiom Beta:** In any adaptive system, long-term viability requires the continuous generation of internal variance.\n\nBoth axioms are empirically supported within their respective scopes and neither may be discarded, weakened, or reinterpreted. Formulate a higher-order systemic framework — a set of structural conditions or a meta-rule — that integrates both axioms without violating the invariants of either. The framework must specify the operational domain, timescale, or system-level condition under which each axiom governs, such that no logical conflict arises. Present the framework in structured prose, clearly labeling the boundary conditions that activate each axiom.\n\n---'}, {'id': 'p14', 'block': 'Stochastic Diversity & Latent Space Expansion', 'concept_name': 'Non-Linear Extrapolation', 'evaluation_target': 'English cross-lingual hardened stress vector: Non-Linear Extrapolation', 'prompt_text': 'The following dataset represents annual reported cases of a network-transmitted anomaly across six consecutive years:\n\n| Year | Cases |\n|------|-------|\n| Y1   | 120   |\n| Y2   | 134   |\n| Y3   | 163   |\n| Y4   | 229   |\n| Y5   | 371   |\n| Y6   | 658   |\n\nLinear extrapolation is strictly forbidden. Identify the underlying growth regime (do not assume linearity — characterize the functional form of the trend), derive the governing growth parameter(s) from the data, and project the case count for Years Y7, Y8, and Y9. Show the derivation of your growth parameter explicitly. Round all projections to the nearest whole number.\n\n---'}, {'id': 'p15', 'block': 'Stochastic Diversity & Latent Space Expansion', 'concept_name': 'Atypical Solution Retrieval', 'evaluation_target': 'English cross-lingual hardened stress vector: Atypical Solution Retrieval', 'prompt_text': "**Optimization problem:** Minimize total latency for routing 1,000 data packets per second through a network that has three active paths: Path 1 (capacity 400 pps, latency 10ms), Path 2 (capacity 400 pps, latency 15ms), and Path 3 (capacity 400 pps, latency 25ms).\n\n**Prohibited solutions:**\n\n1. Weighted round-robin or any static proportional load distribution\n2. Greedy allocation (fill lowest-latency path first until saturated, then overflow)\n3. Any solution that treats the three paths as interchangeable and distributes load uniformly\n\nPropose a valid solution that does not fall into any of the three prohibited categories. State the solution's operational logic, specify the exact packet allocation it produces, calculate the resulting average latency, and explain why it outperforms or matches the theoretical optimum without belonging to any prohibited class.\n\n---\n\n## BLOCK 4 — Complex Correlation & Systemic Stress\n\n---"}, {'id': 'p16', 'block': 'Complex Correlation & Systemic Stress', 'concept_name': 'Game Theory Matrix', 'evaluation_target': 'English cross-lingual hardened stress vector: Game Theory Matrix', 'prompt_text': "Two abstract agents, **Agent R** and **Agent C**, interact in a single-round, non-zero-sum strategic encounter. Agent R has full information about the payoff matrix. Agent C has imperfect information: it knows its own payoffs but assigns a uniform prior over Agent R's two possible strategies.\n\n**Payoff matrix** (format: R's payoff, C's payoff):\n\n|              | C plays X | C plays Y |\n|--------------|-----------|-----------|\n| **R plays 1** | (6, 2)   | (1, 5)   |\n| **R plays 2** | (3, 4)   | (4, 3)   |\n\nIdentify the Nash equilibrium (or equilibria) of this game under the stated information asymmetry. Show the best-response analysis for each agent given their respective information sets. If a pure-strategy Nash equilibrium exists, state it. If only a mixed-strategy equilibrium exists, derive the equilibrium mixing probabilities. Clearly separate the analysis for Agent R (full information) from the analysis for Agent C (imperfect information).\n\n---"}, {'id': 'p17', 'block': 'Complex Correlation & Systemic Stress', 'concept_name': 'Feedback Loop Identification', 'evaluation_target': 'English cross-lingual hardened stress vector: Feedback Loop Identification', 'prompt_text': 'The following closed ecosystem is described by six interacting flow variables:\n\n- **W** (worker population)\n- **R** (resource availability)\n- **P** (production output)\n- **D** (demand pressure)\n- **I** (infrastructure investment)\n- **E** (environmental degradation)\n\nKnown causal relationships (→ indicates "increases", –| indicates "decreases"):\n\n- W → P\n- R → P\n- P → D\n- D → I\n- I → W\n- P –| R\n- P → E\n- E –| W\n- I → R (with a time lag)\n\nFrom this system, isolate and fully trace:\n\n1. **One reinforcing (positive) feedback loop**: identify every node in the loop, the direction of each causal link, and confirm that the net loop polarity is positive.\n2. **One balancing (negative) feedback loop**: same requirements, confirming net negative polarity.\n\nFor each loop, write it as an ordered cycle (e.g., X → Y → Z –| X) and provide the polarity calculation (product of link signs).\n\n---'}, {'id': 'p18', 'block': 'Complex Correlation & Systemic Stress', 'concept_name': 'Bayesian Probabilistic Update', 'evaluation_target': 'English cross-lingual hardened stress vector: Bayesian Probabilistic Update', 'prompt_text': "**Prior:** The probability that a manufactured sensor unit is defective at the time of shipment is P(D) = 0.04.\n\n**Diagnostic test characteristics:** The diagnostic test has a true positive rate (sensitivity) of 0.92 and a false positive rate of 0.08.\n\n**Evidence:** A randomly selected unit from a shipment tests positive on the diagnostic.\n\nUsing Bayes' theorem, compute the posterior probability P(D | positive test). Show all intermediate values: P(positive | D), P(positive | ¬D), P(positive) [total probability], and the final posterior. Express the final answer as a percentage rounded to two decimal places. No approximations are permitted — carry all intermediate values to at least four decimal places before rounding the final output.\n\n---"}, {'id': 'p19', 'block': 'Complex Correlation & Systemic Stress', 'concept_name': 'Cascading Failure Propagation', 'evaluation_target': 'English cross-lingual hardened stress vector: Cascading Failure Propagation', 'prompt_text': 'A simplified network consists of the following nodes and directed dependency links (A → B means B depends on A for operation):\n\n- **HUB** → N1, N2, N3, N4\n- **N1** → S1a, S1b\n- **N2** → S2a\n- **N3** → S3a, S3b, S3c\n- **N4** → S4a\n- **S2a** → T1\n- **S3a** → T1\n- **T1** → T2\n\n**Failure event:** HUB collapses completely at T=0 and cannot be recovered.\n\nConstruct a temporal propagation map with discrete failure rounds. In Round 1, identify all nodes whose sole operational dependency is HUB and that therefore fail immediately. In each subsequent round, identify all nodes whose all upstream dependencies have now failed and that therefore fail in that round. Continue until no further failures propagate. List all nodes that remain fully operational throughout. Format output as a table with columns: `Round | Nodes Failing This Round | Cumulative Failed Set`.\n\n---'}, {'id': 'p20', 'block': 'Complex Correlation & Systemic Stress', 'concept_name': 'Fractal Invariants', 'evaluation_target': 'English cross-lingual hardened stress vector: Fractal Invariants', 'prompt_text': "The following measurements describe a single phenomenon observed at three distinct scales of resolution:\n\n| Scale | Observation |\n|-------|-------------|\n| Macro (10³ units) | 27 discrete structural clusters are present; each cluster contains on average 9 sub-units |\n| Meso (10¹ units)  | 9 discrete structural clusters are present; each cluster contains on average 3 sub-units  |\n| Micro (10⁻¹ units) | 3 discrete structural clusters are present; each cluster contains on average 1 sub-unit  |\n\nFrom these three observations, deduce the generative rule governing the system's self-similar structure across scales. Express the rule as a formal recurrence relation or scaling law. Identify the scaling ratio, the branching factor, and the fractal dimension (using the appropriate definition for this type of structure). State the boundary condition that terminates the recursion. Show all derivations explicitly."}]
BLOCK_TO_CATEGORY = {'Deterministic Containment & State Constraints': 'deterministic_constraint', 'Complete Traceability & Sequential Logic': 'sequential_logic', 'Stochastic Diversity & Latent Space Expansion': 'stochastic_expansion', 'Complex Correlation & Systemic Stress': 'complex_correlation'}
SYSTEM_PROMPT = "You are a precise evaluation model. Follow the user's requested output contract exactly. Return only the requested final artifact. Do not expose reasoning, scratch work, hidden deliberation, or a restatement of the instructions. Stop when the requested artifact is complete."
EXPECTED_PROMPT_SHA256 = '4d4770248cecb07384f451dca107bef3ae7201a4513a67d87dd90abd660729ec'
EXPECTED_SYSTEM_SHA256 = 'e20a3a96f983a9d6b75a1ebaa669a36cd15bcf7aeb1ab2f1c52d2959312f10cb'


def stable_hash(value):
    raw = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


canonical_prompts = [
    {
        "id": row["id"],
        "block": row["block"],
        "concept_name": row["concept_name"],
        "evaluation_target": row["evaluation_target"],
        "prompt_text": row["prompt_text"],
    }
    for row in PROMPTS_RAW
]

assert stable_hash(canonical_prompts) == EXPECTED_PROMPT_SHA256
assert hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest() == EXPECTED_SYSTEM_SHA256

PROMPT_RECORDS = []
for row in PROMPTS_RAW:
    prompt_id = row["id"]
    exclusion_reason = CONFIRMATORY_EXCLUSIONS.get(prompt_id, "")
    PROMPT_RECORDS.append(
        {
            "prompt_id": prompt_id,
            "task_type": BLOCK_TO_CATEGORY[row["block"]],
            "block": row["block"],
            "concept_name": row["concept_name"],
            "evaluation_target": row["evaluation_target"],
            "prompt_text": row["prompt_text"],
            "prompt_sha256": hashlib.sha256(
                row["prompt_text"].encode()
            ).hexdigest(),
            "analysis_eligible": prompt_id not in CONFIRMATORY_EXCLUSIONS,
            "analysis_tier": (
                "confirmatory"
                if prompt_id not in CONFIRMATORY_EXCLUSIONS
                else "exploratory_excluded"
            ),
            "analysis_exclusion_reason": exclusion_reason,
        }
    )

CONFIRMATORY_PROMPT_IDS = [
    row["prompt_id"] for row in PROMPT_RECORDS if row["analysis_eligible"]
]
EXPLORATORY_EXCLUDED_PROMPT_IDS = [
    row["prompt_id"] for row in PROMPT_RECORDS if not row["analysis_eligible"]
]

PROMPT_MANIFEST_SHA256 = stable_hash(PROMPT_RECORDS)
prompt_manifest = pd.DataFrame(PROMPT_RECORDS)
prompt_manifest.to_csv(
    CAMPAIGN_ROOT / "prompt_manifest.csv",
    index=False,
)

print("Locked battery hash:", EXPECTED_PROMPT_SHA256)
print("Prompt manifest hash:", PROMPT_MANIFEST_SHA256)
print("Confirmatory prompts:", len(CONFIRMATORY_PROMPT_IDS))
print("Exploratory excluded prompts:", EXPLORATORY_EXCLUDED_PROMPT_IDS)




In [ ]:
# ============================================================
# 03. RECONSTRUCT AND VERIFY EMBEDDED RUNTIME v3.8
# ============================================================
import hashlib, importlib, io, json, shutil, sys, zipfile

EMBEDDED_RUNTIME_HEX = '...'
EXPECTED_RUNTIME_SHA256 = 'f5d939ec46cffbbfb51e62bc0412b060b6868ae931ff03a31884998eb0bc4d9a'

runtime_payload = bytes.fromhex(EMBEDDED_RUNTIME_HEX)
actual_runtime_sha256 = hashlib.sha256(runtime_payload).hexdigest()
assert actual_runtime_sha256 == EXPECTED_RUNTIME_SHA256
RUNTIME_DIR = CAMPAIGN_ROOT / "runtime"
if RUNTIME_DIR.exists(): shutil.rmtree(RUNTIME_DIR)
RUNTIME_DIR.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(runtime_payload)) as archive:
    assert archive.testzip() is None
    archive.extractall(RUNTIME_DIR)
runtime_manifest = json.loads((RUNTIME_DIR / "RUNTIME_MANIFEST.json").read_text(encoding="utf-8"))
for entry in runtime_manifest["files"]:
    path = RUNTIME_DIR / entry["path"]
    assert path.exists(), entry["path"]
    assert path.stat().st_size == entry["bytes"]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == entry["sha256"]
sys.path.insert(0, str(RUNTIME_DIR))
for name in list(sys.modules):
    if name == "rds_v35" or name.startswith("rds_v35."):
        del sys.modules[name]
importlib.invalidate_caches()
import rds_v35
assert rds_v35.__version__ == "3.8.0", rds_v35.__version__
print("Runtime SHA-256:", actual_runtime_sha256)
print("Runtime version:", rds_v35.__version__)
print("Runtime files verified:", len(runtime_manifest["files"]))


In [ ]:
# ============================================================
# 04. RUNTIME TESTS + SCIENTIFIC INVARIANTS
# ============================================================
import os, subprocess, sys
from rds_v35 import (
    GENERATION_ACTIONS, ROUTER_ACTIONS, PROBE_ACTIONS,
    QASConfig, PostWarperRiskAwareRouter,
)

assert len(GENERATION_ACTIONS) == 7
assert len(PROBE_ACTIONS) == 6
assert set(ROUTER_ACTIONS).issubset(GENERATION_ACTIONS)
assert ENABLE_EVENT_REROUTING is False

test_environment = os.environ.copy()
test_environment["PYTHONPATH"] = str(RUNTIME_DIR)
test_environment["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
test_process = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", str(RUNTIME_DIR / "tests")],
    text=True, capture_output=True, env=test_environment, timeout=600,
)
print(test_process.stdout)
if test_process.returncode != 0:
    print(test_process.stderr)
    raise RuntimeError("Embedded runtime tests failed.")

router = PostWarperRiskAwareRouter(**ROUTER_PARAMETERS)
QAS_CONFIG = QASConfig(**QAS_PARAMETERS)
print("Router actions:", ROUTER_ACTIONS)
print("Generation actions:", GENERATION_ACTIONS)
print("Probe actions:", PROBE_ACTIONS)


In [ ]:
# 05. OFFICIAL CONFIG-DRIVEN QWEN3.5 LOADING
# 4-BIT NF4 • TEXT-ONLY INPUT • NON-THINKING
# ============================================================
import gc
import json
import re

import torch
import transformers
from transformers import AutoConfig, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available in this runtime.")
if tuple(torch.cuda.get_device_capability(0)) == (6, 0):
    if "sm_60" not in torch.cuda.get_arch_list():
        raise RuntimeError("The verified Torch build does not contain sm_60.")

TRANSFORMERS_VERSION = transformers.__version__
if TRANSFORMERS_VERSION != "5.12.1":
    raise RuntimeError(
        f"Unexpected Transformers version: {TRANSFORMERS_VERSION}; expected 5.12.1"
    )

MODEL_CONFIG = AutoConfig.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION_RESOLVED,
    token=HF_TOKEN,
    trust_remote_code=False,
)

MODEL_CONFIG_MODEL_CLASSES = list(
    getattr(MODEL_CONFIG, "arch" + "itectures", None)
    or STACK_INFO.get("config_model_classes")
    or []
)
MODEL_CONFIG_TYPE = str(getattr(MODEL_CONFIG, "model_type", ""))
MODEL_HAS_VISION = bool(
    getattr(MODEL_CONFIG, "vision_config", None) is not None
    or STACK_INFO.get("config_has_vision", False)
)
MODEL_BACKEND_KIND = "multimodal" if MODEL_HAS_VISION else "causal_lm"

print("Transformers:", TRANSFORMERS_VERSION)
print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Config model classes:", MODEL_CONFIG_MODEL_CLASSES)
print("Config model_type:", MODEL_CONFIG_TYPE)
print("Config has vision:", MODEL_HAS_VISION)
print("Selected backend:", MODEL_BACKEND_KIND)

if MODEL_BACKEND_KIND == "multimodal":
    AutoProcessor = getattr(transformers, "AutoProcessor", None)
    if AutoProcessor is None:
        raise RuntimeError("AutoProcessor is unavailable for the multimodal checkpoint.")

    chat_handler = AutoProcessor.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION_RESOLVED,
        token=HF_TOKEN,
        trust_remote_code=False,
    )
    tokenizer = getattr(chat_handler, "tokenizer", None)
    if tokenizer is None:
        raise RuntimeError("AutoProcessor did not expose its tokenizer.")
    CHAT_HANDLER_USED = "AutoProcessor"
else:
    AutoTokenizer = getattr(transformers, "AutoTokenizer", None)
    if AutoTokenizer is None:
        raise RuntimeError("AutoTokenizer is unavailable.")

    chat_handler = AutoTokenizer.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION_RESOLVED,
        token=HF_TOKEN,
        use_fast=True,
        trust_remote_code=False,
    )
    tokenizer = chat_handler
    CHAT_HANDLER_USED = "AutoTokenizer"

processor = chat_handler

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
if not getattr(chat_handler, "chat_template", None):
    if not getattr(tokenizer, "chat_template", None):
        raise RuntimeError("The Qwen3.5 chat template was not found.")

# Official automatic class first, exact model class second, generic fallback last.
model_class_candidates = []

if MODEL_BACKEND_KIND == "multimodal":
    for class_name in (
        "AutoModelForMultimodalLM",
        *MODEL_CONFIG_MODEL_CLASSES,
        "AutoModelForImageTextToText",
    ):
        model_class = getattr(transformers, class_name, None)
        if model_class is not None:
            model_class_candidates.append((class_name, model_class))
else:
    for class_name in (
        "AutoModelForCausalLM",
        *MODEL_CONFIG_MODEL_CLASSES,
    ):
        model_class = getattr(transformers, class_name, None)
        if model_class is not None:
            model_class_candidates.append((class_name, model_class))

deduplicated = []
seen = set()
for name, cls in model_class_candidates:
    if name not in seen:
        deduplicated.append((name, cls))
        seen.add(name)
model_class_candidates = deduplicated

if not model_class_candidates:
    raise RuntimeError("No model class is compatible with the resolved config.")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

load_variants = [
    {
        "quantization_config": quantization_config,
        "device_map": {"": 0},
        "dtype": torch.float16,
        "low_cpu_mem_usage": True,
        "attn_implementation": "sdpa",
    },
    {
        "quantization_config": quantization_config,
        "device_map": {"": 0},
        "torch_dtype": torch.float16,
        "low_cpu_mem_usage": True,
        "attn_implementation": "sdpa",
    },
    {
        "quantization_config": quantization_config,
        "device_map": {"": 0},
        "torch_dtype": torch.float16,
        "low_cpu_mem_usage": True,
    },
]

model = None
MODEL_CLASS_USED = None
load_errors = []

for class_name, model_class in model_class_candidates:
    for attempt_index, kwargs in enumerate(load_variants, 1):
        try:
            print(f"Load {class_name}, attempt {attempt_index}")
            model = model_class.from_pretrained(
                MODEL_ID,
                revision=MODEL_REVISION_RESOLVED,
                token=HF_TOKEN,
                config=MODEL_CONFIG,
                trust_remote_code=False,
                **kwargs,
            )
            MODEL_CLASS_USED = class_name
            break
        except Exception as exc:
            load_errors.append(
                {
                    "class": class_name,
                    "attempt": attempt_index,
                    "error": repr(exc),
                }
            )
            gc.collect()
            torch.cuda.empty_cache()
    if model is not None:
        break

if model is None:
    raise RuntimeError(
        "Every native 4-bit model-loading attempt failed:\n"
        + json.dumps(load_errors, indent=2)
    )

model.eval()

MODEL_COMPATIBILITY_AUDIT = {
    "model_id": MODEL_ID,
    "resolved_revision": MODEL_REVISION_RESOLVED,
    "config_model_classes": MODEL_CONFIG_MODEL_CLASSES,
    "config_model_type": MODEL_CONFIG_TYPE,
    "config_has_vision": MODEL_HAS_VISION,
    "backend_kind": MODEL_BACKEND_KIND,
    "model_class_used": MODEL_CLASS_USED,
    "chat_handler_used": CHAT_HANDLER_USED,
    "transformers_version": TRANSFORMERS_VERSION,
    "torch_version": torch.__version__,
    "trust_remote_code": False,
    "load_errors_before_success": load_errors,
}
(CAMPAIGN_ROOT / "model_compatibility_audit.json").write_text(
    json.dumps(MODEL_COMPATIBILITY_AUDIT, indent=2),
    encoding="utf-8",
)

from rds_v35 import (
    decode_candidate,
    model_device,
    move_encoded,
    render_messages,
)

rendered_smoke, encoded_smoke = render_messages(
    chat_handler,
    SYSTEM_PROMPT,
    "Reply with exactly: OK",
)

open_tags = rendered_smoke.count("<think>")
close_tags = rendered_smoke.count("</think>")
if open_tags != close_tags:
    raise RuntimeError("Unbalanced think tags in the rendered non-thinking template.")

if open_tags:
    think_blocks = re.findall(
        r"<think>(.*?)</think>",
        rendered_smoke,
        flags=re.S,
    )
    if any(block.strip() for block in think_blocks):
        raise RuntimeError(
            "The supposedly non-thinking template contains non-empty reasoning."
        )

encoded_smoke = move_encoded(encoded_smoke, model_device(model))
smoke = decode_candidate(
    model,
    tokenizer,
    encoded_smoke,
    action="baseline",
    seed=11,
    max_new_tokens=32,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    preserve_top1=True,
    variant="baseline",
)

print("Model class used:", MODEL_CLASS_USED)
print("Chat handler used:", CHAT_HANDLER_USED)
print("Smoke raw:", repr(smoke.raw_text))
print("Smoke final:", repr(smoke.final_text))

if smoke.think_tag_present:
    raise RuntimeError("Thinking tags appeared in the non-thinking smoke output.")
if not smoke.final_text.strip():
    raise RuntimeError("The smoke output is empty.")

del encoded_smoke
gc.collect()
torch.cuda.empty_cache()

print("✅ Native non-thinking Qwen3.5 smoke test passed.")




In [ ]:
# ============================================================
# 06. EX-ANTE ROUTE PLAN + POST-WARPER PROBE MAP
# ============================================================
import gc, json, pandas as pd, torch
from rds_v35 import capture_prefill_logits, model_device, move_encoded, render_messages

ROUTE_DECISIONS = {}
route_rows = []
fingerprint_rows = []

for spec in PROMPT_RECORDS:
    prompt_id = spec["prompt_id"]
    task_type = spec["task_type"]
    sampling = SAMPLING_BY_TASK[task_type]
    _, encoded = render_messages(chat_handler, SYSTEM_PROMPT, spec["prompt_text"])
    encoded = move_encoded(encoded, model_device(model))
    base_logits = capture_prefill_logits(model, encoded)
    decision = router.route(
        prompt_id, task_type, spec["prompt_text"], base_logits,
        temperature=sampling["temperature"], top_k=sampling["top_k"], top_p=sampling["top_p"],
    )
    ROUTE_DECISIONS[prompt_id] = decision
    route_rows.append({
        "prompt_id": prompt_id,
        "task_type": task_type,
        "analysis_eligible": spec["analysis_eligible"],
        "selected_action": decision.selected_action,
        "control_action": decision.control_action,
        "abstained": decision.abstained,
        "reason": decision.reason,
        "discrimination_margin": decision.discrimination_margin,
        "stress_risk": decision.stress_risk,
        "micro_response": decision.micro_response,
        "temperature": sampling["temperature"],
        "top_p": sampling["top_p"],
        "top_k": sampling["top_k"],
        "scores_json": json.dumps(decision.scores),
        "score_components_json": json.dumps(decision.score_components),
    })
    for action, fp in decision.fingerprints.items():
        fingerprint_rows.append({"prompt_id": prompt_id, "task_type": task_type, "action": action, **fp})
    del encoded, base_logits
    gc.collect(); torch.cuda.empty_cache()

route_plan = pd.DataFrame(route_rows)
probe_fingerprints = pd.DataFrame(fingerprint_rows)
route_plan.to_csv(CAMPAIGN_ROOT / "route_plan.csv", index=False)
probe_fingerprints.to_csv(CAMPAIGN_ROOT / "probe_fingerprints.csv", index=False)

print(route_plan[["prompt_id","task_type","selected_action","control_action","abstained"]].to_string(index=False))
print("Selected distribution:", route_plan["selected_action"].value_counts().to_dict())
print("Control distribution:", route_plan["control_action"].value_counts().to_dict())


In [ ]:
# ============================================================
# 07. TRANSACTIONAL FIVE-ROLE CAMPAIGN
# baseline + router full/QAS + distinct active-control full/QAS
# ============================================================
from datetime import datetime, timezone
import gc, hashlib, json, os, sqlite3, traceback, zipfile
import pandas as pd, torch
from rds_v35 import decode_candidate, model_device, move_encoded, render_messages
from rds_v35.verifiers import verify_output

DB_PATH = CAMPAIGN_ROOT / "campaign.sqlite"
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA journal_mode=WAL")
conn.execute("PRAGMA synchronous=FULL")
conn.execute("PRAGMA foreign_keys=ON")
conn.executescript("""
CREATE TABLE IF NOT EXISTS generations (
 run_key TEXT PRIMARY KEY, prompt_id TEXT NOT NULL, seed INTEGER NOT NULL,
 variant TEXT NOT NULL, role TEXT NOT NULL, action TEXT NOT NULL, task_type TEXT NOT NULL,
 temperature REAL NOT NULL, top_p REAL NOT NULL, top_k INTEGER NOT NULL,
 raw_text TEXT NOT NULL, final_text TEXT NOT NULL, raw_sha256 TEXT NOT NULL, text_sha256 TEXT NOT NULL,
 token_count INTEGER NOT NULL, generation_seconds REAL NOT NULL,
 ended_with_eos INTEGER NOT NULL, hit_max_new_tokens INTEGER NOT NULL,
 termination_reason TEXT NOT NULL, meta_contamination INTEGER NOT NULL, think_tag_present INTEGER NOT NULL,
 pre_projection_flips INTEGER NOT NULL, post_projection_flips INTEGER NOT NULL,
 exact_pass INTEGER NOT NULL, format_pass INTEGER NOT NULL, content_pass INTEGER NOT NULL,
 terminal_pass INTEGER NOT NULL, automatic_score REAL, scoreable INTEGER NOT NULL,
 qas_mean_mask_size REAL, qas_mean_mask_fraction REAL,
 qas_retained_selected_energy REAL, qas_retained_support_energy REAL,
 qas_target_reached_rate REAL, qas_candidate_cap_hit_rate REAL, qas_mask_cap_hit_rate REAL,
 qas_mean_postwarper_tv REAL, qas_mean_postwarper_kl REAL,
 qas_mean_jaccard REAL, qas_phase_events INTEGER,
 verification_json TEXT NOT NULL, diagnostics_json TEXT NOT NULL, sketches_json TEXT NOT NULL,
 created_utc TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS errors (
 run_key TEXT, prompt_id TEXT, seed INTEGER, variant TEXT, role TEXT, action TEXT,
 attempt INTEGER, error TEXT, traceback TEXT, created_utc TEXT
);
CREATE TABLE IF NOT EXISTS metadata (key TEXT PRIMARY KEY, value TEXT NOT NULL);
""")
conn.commit()

metadata = {
 "software_version": SOFTWARE_VERSION, "model_id": MODEL_ID, "model_family": MODEL_FAMILY,
 "model_class": MODEL_CLASS_USED, "model_revision_resolved": MODEL_REVISION_RESOLVED,
 "protocol_id": PROTOCOL_ID, "experiment_id": EXPERIMENT_ID, "max_new_tokens": MAX_NEW_TOKENS,
 "seeds": SEEDS, "prompt_manifest_sha256": PROMPT_MANIFEST_SHA256,
 "system_prompt_sha256": EXPECTED_SYSTEM_SHA256, "runtime_sha256": EXPECTED_RUNTIME_SHA256,
 "router_parameters": ROUTER_PARAMETERS, "qas_parameters": QAS_PARAMETERS,
 "sampling_by_task": SAMPLING_BY_TASK, "active_control_enabled": ENABLE_ACTIVE_CONTROL,
 "event_rerouting_enabled": ENABLE_EVENT_REROUTING,
}
for k,v in metadata.items(): conn.execute("INSERT OR REPLACE INTO metadata(key,value) VALUES (?,?)",(k,json.dumps(v,default=str)))
conn.commit()

def success_count(): return int(conn.execute("SELECT COUNT(*) FROM generations").fetchone()[0])
def has_result(key): return conn.execute("SELECT 1 FROM generations WHERE run_key=?",(key,)).fetchone() is not None

def export_current_tables():
    g=pd.read_sql_query("SELECT * FROM generations ORDER BY prompt_id,seed,variant",conn)
    e=pd.read_sql_query("SELECT * FROM errors ORDER BY created_utc",conn)
    m=pd.read_sql_query("SELECT * FROM metadata ORDER BY key",conn)
    g.to_csv(CAMPAIGN_ROOT/"full_results.csv",index=False)
    g.to_json(CAMPAIGN_ROOT/"full_results.jsonl",orient="records",lines=True,force_ascii=False)
    e.to_csv(CAMPAIGN_ROOT/"errors.csv",index=False); m.to_csv(CAMPAIGN_ROOT/"run_metadata.csv",index=False)
    with (CAMPAIGN_ROOT/"trajectory_sketches.jsonl").open("w",encoding="utf-8") as h:
        for _,row in g.iterrows():
            for sketch in json.loads(row["sketches_json"]):
                h.write(json.dumps({"run_key":row["run_key"],"prompt_id":row["prompt_id"],"seed":int(row["seed"]),"variant":row["variant"],"role":row["role"],"action":row["action"],**sketch},ensure_ascii=False)+"\n")
    return g

def checkpoint_files():
    return ["campaign.sqlite","full_results.csv","full_results.jsonl","trajectory_sketches.jsonl","errors.csv","run_metadata.csv","route_plan.csv","probe_fingerprints.csv","prompt_manifest.csv","model_compatibility_audit.json"]

def write_checkpoint(count):
    conn.execute("PRAGMA wal_checkpoint(FULL)"); conn.commit(); export_current_tables()
    target=WORK_ROOT/f"{EXPERIMENT_ID}_CHECKPOINT_{count:04d}.zip"; partial=target.with_suffix(".zip.partial")
    if partial.exists(): partial.unlink()
    with zipfile.ZipFile(partial,"w",zipfile.ZIP_DEFLATED,allowZip64=True) as z:
        for name in checkpoint_files():
            p=CAMPAIGN_ROOT/name
            if p.exists(): z.write(p,name)
    with zipfile.ZipFile(partial) as z: assert z.testzip() is None
    os.replace(partial,target)
    print("✅ CHECKPOINT",target,"SHA256",hashlib.sha256(target.read_bytes()).hexdigest())

for spec in PROMPT_RECORDS:
    pid=spec["prompt_id"]; task=spec["task_type"]; sampling=SAMPLING_BY_TASK[task]
    decision=ROUTE_DECISIONS[pid]
    _,encoded=render_messages(chat_handler,SYSTEM_PROMPT,spec["prompt_text"])
    encoded=move_encoded(encoded,model_device(model))
    for seed in SEEDS:
        jobs=[("baseline","baseline","baseline")]
        if decision.selected_action != "baseline":
            jobs += [("router_full","router_full",decision.selected_action),("router_qas","router_qas",decision.selected_action)]
        if ENABLE_ACTIVE_CONTROL and decision.control_action != "baseline" and decision.control_action != decision.selected_action:
            jobs += [("control_full","control_full",decision.control_action),("control_qas","control_qas",decision.control_action)]
        for variant,role,action in jobs:
            key=f"{EXPERIMENT_ID}|{pid}|{seed}|{variant}|{action}"
            if has_result(key): print("SKIP",key); continue
            for attempt in range(1,MAX_TRANSIENT_RETRIES+2):
                stamp=datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
                try:
                    cand=decode_candidate(model,tokenizer,encoded,action=action,seed=seed,max_new_tokens=MAX_NEW_TOKENS,
                        temperature=sampling["temperature"],top_p=sampling["top_p"],top_k=sampling["top_k"],
                        preserve_top1=PRESERVE_TOP1,variant=variant,qas_config=QAS_CONFIG)
                    if cand.post_projection_flips != 0: raise RuntimeError(f"Top-1 projection failed: {key}")
                    ver=verify_output(pid,cand.final_text)
                    terminal=bool(cand.ended_with_eos and not cand.meta_contamination and cand.final_text.strip())
                    ver["terminal_pass"]=terminal; ver["completion_pass"]=terminal
                    scoreable=bool(ver.get("scoreable",False))
                    auto=(sum([int(bool(ver.get("exact_pass"))),int(bool(ver.get("format_pass"))),int(bool(ver.get("content_pass"))),int(terminal)])/4.0) if scoreable else None
                    d=cand.operator_diagnostics
                    conn.execute("""INSERT OR REPLACE INTO generations VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",(
                        key,pid,seed,variant,role,action,task,sampling["temperature"],sampling["top_p"],sampling["top_k"],
                        cand.raw_text,cand.final_text,hashlib.sha256(cand.raw_text.encode()).hexdigest(),hashlib.sha256(cand.final_text.encode()).hexdigest(),
                        cand.token_count,cand.generation_seconds,int(cand.ended_with_eos),int(cand.hit_max_new_tokens),cand.termination_reason,
                        int(cand.meta_contamination),int(cand.think_tag_present),cand.pre_projection_flips,cand.post_projection_flips,
                        int(bool(ver.get("exact_pass"))),int(bool(ver.get("format_pass"))),int(bool(ver.get("content_pass"))),int(terminal),auto,int(scoreable),
                        d.get("mean_mask_size"),d.get("mean_mask_fraction"),d.get("mean_retained_selected_operator_energy"),d.get("mean_retained_support_energy"),
                        d.get("target_reached_rate"),d.get("candidate_topk_cap_hit_rate"),d.get("max_mask_cap_hit_rate"),d.get("mean_postwarper_tv"),d.get("mean_postwarper_kl"),
                        d.get("mean_mask_jaccard"),d.get("phase_events"),json.dumps(ver,ensure_ascii=False),json.dumps(d,ensure_ascii=False),json.dumps(cand.trajectory_sketches,ensure_ascii=False),stamp
                    ))
                    conn.commit(); count=success_count()
                    print(f"[{count:03d}] {pid} seed={seed} {variant}:{action} tokens={cand.token_count} stop={cand.termination_reason} score={auto}")
                    if CHECKPOINT_EVERY_GENERATIONS and count%CHECKPOINT_EVERY_GENERATIONS==0: write_checkpoint(count)
                    break
                except Exception as exc:
                    conn.execute("INSERT INTO errors VALUES (?,?,?,?,?,?,?,?,?,?)",(key,pid,seed,variant,role,action,attempt,repr(exc),traceback.format_exc(),stamp)); conn.commit()
                    print("ERROR",key,repr(exc),"attempt",attempt)
                    if attempt>MAX_TRANSIENT_RETRIES: raise
    del encoded; gc.collect(); torch.cuda.empty_cache()

CURRENT_RESULTS=export_current_tables(); write_checkpoint(success_count())
print("✅ Campaign complete",len(CURRENT_RESULTS),"rows")


In [ ]:
# ============================================================
# 08. CONFIRMATORY ANALYSIS + BLIND PAIRS + FINAL CUSTODY
# ============================================================
import hashlib, json, math, random, zipfile
import numpy as np, pandas as pd

full=pd.read_sql_query("SELECT * FROM generations ORDER BY prompt_id,seed,variant",conn)
routes=pd.read_csv(CAMPAIGN_ROOT/"route_plan.csv")
prompts=pd.DataFrame(PROMPT_RECORDS)
eligible=dict(zip(prompts.prompt_id,prompts.analysis_eligible.astype(bool)))

contrast_specs=[
 ("router_full_vs_baseline","baseline","router_full"),
 ("router_qas_vs_baseline","baseline","router_qas"),
 ("router_qas_vs_router_full","router_full","router_qas"),
 ("control_full_vs_baseline","baseline","control_full"),
 ("control_qas_vs_baseline","baseline","control_qas"),
 ("control_qas_vs_control_full","control_full","control_qas"),
]
rows=[]
for pid in sorted(full.prompt_id.unique()):
    if not eligible.get(pid,False): continue
    for seed in SEEDS:
        group=full[(full.prompt_id==pid)&(full.seed==seed)].set_index("variant")
        for contrast,refv,candv in contrast_specs:
            if refv not in group.index or candv not in group.index: continue
            ref=group.loc[refv]; cand=group.loc[candv]
            both_scoreable=bool(ref.scoreable and cand.scoreable)
            row={"contrast":contrast,"prompt_id":pid,"seed":seed,"task_type":ref.task_type,
                 "reference_variant":refv,"candidate_variant":candv,"reference_action":ref.action,"candidate_action":cand.action,
                 "texts_identical":ref.final_text==cand.final_text,"raw_texts_identical":ref.raw_text==cand.raw_text,
                 "delta_terminal":cand.terminal_pass-ref.terminal_pass,"delta_tokens":cand.token_count-ref.token_count,
                 "delta_hit_max":cand.hit_max_new_tokens-ref.hit_max_new_tokens,"delta_meta":cand.meta_contamination-ref.meta_contamination,
                 "both_scoreable":both_scoreable}
            for metric,col in [("exact","exact_pass"),("format","format_pass"),("content","content_pass"),("automatic_score","automatic_score")]:
                row[f"delta_{metric}"]=(cand[col]-ref[col]) if both_scoreable and pd.notna(cand[col]) and pd.notna(ref[col]) else np.nan
            rows.append(row)
paired=pd.DataFrame(rows); paired.to_csv(CAMPAIGN_ROOT/"paired_contrasts.csv",index=False)

sample_sizes=(paired.groupby("contrast").agg(n_rows=("prompt_id","size"),n_prompts=("prompt_id","nunique"),divergent_pairs=("texts_identical",lambda s:int((~s).sum())),scoreable_pairs=("both_scoreable","sum")).reset_index())
sample_sizes.to_csv(CAMPAIGN_ROOT/"contrast_sample_sizes.csv",index=False)

# Method summaries: semantic rates only over scoreable rows; terminal and throughput over all rows.
summary=[]
for (variant,action,task),g in full[full.prompt_id.map(eligible).fillna(False)].groupby(["variant","action","task_type"],dropna=False):
    sg=g[g.scoreable==1]
    summary.append({"variant":variant,"action":action,"task_type":task,"n":len(g),"n_scoreable":len(sg),
        "exact_rate":sg.exact_pass.mean() if len(sg) else np.nan,"format_rate":sg.format_pass.mean() if len(sg) else np.nan,
        "content_rate":sg.content_pass.mean() if len(sg) else np.nan,"mean_automatic_score":sg.automatic_score.mean() if len(sg) else np.nan,
        "terminal_rate":g.terminal_pass.mean(),"mean_tokens":g.token_count.mean(),"tokens_per_second":g.token_count.sum()/g.generation_seconds.sum(),
        "mean_qas_mask_size":g.qas_mean_mask_size.mean(),"mean_qas_mask_fraction":g.qas_mean_mask_fraction.mean(),
        "mean_retained_selected_energy":g.qas_retained_selected_energy.mean(),"mean_retained_support_energy":g.qas_retained_support_energy.mean(),
        "target_reached_rate":g.qas_target_reached_rate.mean(),"mean_qas_postwarper_tv":g.qas_mean_postwarper_tv.mean()})
method_summary=pd.DataFrame(summary); method_summary.to_csv(CAMPAIGN_ROOT/"method_summary.csv",index=False)

metrics=["delta_exact","delta_format","delta_content","delta_automatic_score","delta_terminal","delta_tokens","delta_hit_max","delta_meta"]
boot_rows=[]; corr_rows=[]
for contrast,g in paired.groupby("contrast"):
    pids=sorted(g.prompt_id.unique()); seed_offset=int(hashlib.sha256(contrast.encode()).hexdigest()[:8],16)
    rng=np.random.default_rng(1701+seed_offset%10000)
    draw_matrix={m:[] for m in metrics}
    observed={m:g[m].mean(skipna=True) for m in metrics}
    for _ in range(BOOTSTRAP_REPLICATES):
        sampled=rng.choice(pids,size=len(pids),replace=True)
        bg=pd.concat([g[g.prompt_id==pid] for pid in sampled],ignore_index=True)
        for m in metrics: draw_matrix[m].append(bg[m].mean(skipna=True))
    for m in metrics:
        arr=np.asarray(draw_matrix[m],float); finite=arr[np.isfinite(arr)]
        boot_rows.append({"contrast":contrast,"metric":m,"n_prompts":len(pids),"n_rows":len(g),"observed_mean":observed[m],
            "ci_low":np.quantile(finite,0.025) if len(finite) else np.nan,"ci_high":np.quantile(finite,0.975) if len(finite) else np.nan,
            "bootstrap_replicates":BOOTSTRAP_REPLICATES,"estimable":bool(len(finite) and np.nanstd(finite)>0)})
    for i,mx in enumerate(metrics):
        for my in metrics[i+1:]:
            x=np.asarray(draw_matrix[mx],float); y=np.asarray(draw_matrix[my],float); mask=np.isfinite(x)&np.isfinite(y)
            estimable=bool(mask.sum()>2 and np.std(x[mask])>0 and np.std(y[mask])>0)
            corr=float(np.corrcoef(x[mask],y[mask])[0,1]) if estimable else np.nan
            corr_rows.append({"contrast":contrast,"metric_x":mx,"metric_y":my,"bootstrap_correlation":corr,"estimable":estimable,"reason":"" if estimable else "zero_or_missing_variance"})
bootstrap=pd.DataFrame(boot_rows); correlations=pd.DataFrame(corr_rows)
bootstrap.to_csv(CAMPAIGN_ROOT/"bootstrap_ci_global.csv",index=False)
correlations.to_csv(CAMPAIGN_ROOT/"bootstrap_joint_correlations_global.csv",index=False)

# Segment-level bootstrap with the same estimator and explicit non-estimability.
segment_boot=[]; segment_corr=[]
for (task_type,contrast),g in paired.groupby(["task_type","contrast"]):
    pids=sorted(g.prompt_id.unique())
    seed_offset=int(hashlib.sha256(f"{task_type}|{contrast}".encode()).hexdigest()[:8],16)
    rng=np.random.default_rng(3701+seed_offset%10000)
    draw_matrix={m:[] for m in metrics}
    observed={m:g[m].mean(skipna=True) for m in metrics}
    for _ in range(BOOTSTRAP_REPLICATES):
        sampled=rng.choice(pids,size=len(pids),replace=True)
        bg=pd.concat([g[g.prompt_id==pid] for pid in sampled],ignore_index=True)
        for m in metrics: draw_matrix[m].append(bg[m].mean(skipna=True))
    for m in metrics:
        arr=np.asarray(draw_matrix[m],float); finite=arr[np.isfinite(arr)]
        segment_boot.append({"task_type":task_type,"contrast":contrast,"metric":m,"n_prompts":len(pids),"n_rows":len(g),
            "observed_mean":observed[m],"ci_low":np.quantile(finite,0.025) if len(finite) else np.nan,
            "ci_high":np.quantile(finite,0.975) if len(finite) else np.nan,"bootstrap_replicates":BOOTSTRAP_REPLICATES,
            "estimable":bool(len(finite) and np.nanstd(finite)>0)})
    for i,mx in enumerate(metrics):
        for my in metrics[i+1:]:
            x=np.asarray(draw_matrix[mx],float); y=np.asarray(draw_matrix[my],float); mask=np.isfinite(x)&np.isfinite(y)
            estimable=bool(mask.sum()>2 and np.std(x[mask])>0 and np.std(y[mask])>0)
            segment_corr.append({"task_type":task_type,"contrast":contrast,"metric_x":mx,"metric_y":my,
                "bootstrap_correlation":float(np.corrcoef(x[mask],y[mask])[0,1]) if estimable else np.nan,
                "estimable":estimable,"reason":"" if estimable else "zero_or_missing_variance"})
pd.DataFrame(segment_boot).to_csv(CAMPAIGN_ROOT/"bootstrap_ci_segment.csv",index=False)
pd.DataFrame(segment_corr).to_csv(CAMPAIGN_ROOT/"bootstrap_joint_correlations_segment.csv",index=False)

# Blind Human Eval only for divergent outputs.
blind=[]; key=[]; rnd=random.Random(9917)
for i,row in paired[~paired.texts_identical].reset_index(drop=True).iterrows():
    ref=full[(full.prompt_id==row.prompt_id)&(full.seed==row.seed)&(full.variant==row.reference_variant)].iloc[0]
    cand=full[(full.prompt_id==row.prompt_id)&(full.seed==row.seed)&(full.variant==row.candidate_variant)].iloc[0]
    swap=bool(rnd.getrandbits(1)); A=cand if swap else ref; B=ref if swap else cand
    pair_id=f"pair_{i+1:05d}"
    blind.append({"pair_id":pair_id,"prompt_id":row.prompt_id,"seed":row.seed,"contrast":row.contrast,"response_A":A.final_text,"response_B":B.final_text,
        "winner":"","correctness_A":"","correctness_B":"","completeness_A":"","completeness_B":"","coherence_A":"","coherence_B":"","epistemic_A":"","epistemic_B":"","confidence":"","justification":""})
    key.append({"pair_id":pair_id,"A_variant":A.variant,"A_action":A.action,"B_variant":B.variant,"B_action":B.action})
blind_columns=["pair_id","prompt_id","seed","contrast","response_A","response_B","winner","correctness_A","correctness_B","completeness_A","completeness_B","coherence_A","coherence_B","epistemic_A","epistemic_B","confidence","justification"]
key_columns=["pair_id","A_variant","A_action","B_variant","B_action"]
pd.DataFrame(blind,columns=blind_columns).to_csv(CAMPAIGN_ROOT/"human_eval_blind_pairs.csv",index=False)
pd.DataFrame(key,columns=key_columns).to_csv(CAMPAIGN_ROOT/"human_eval_blind_key.csv",index=False)

# Rendered report with actual values.
route_dist=routes.selected_action.value_counts().to_dict(); control_dist=routes.control_action.value_counts().to_dict()
report=f"""# RDS v3.5 — TALM/TALON Post-Warper Activation and QAS Fidelity campaign report

## Population
- Model: `{MODEL_ID}`
- Revision: `{MODEL_REVISION_RESOLVED}`
- Rows: {len(full)}
- Prompts: {full.prompt_id.nunique()}
- Seeds: {SEEDS}
- Router distribution: `{route_dist}`
- Active-control distribution: `{control_dist}`

## Contrasts
```
{sample_sizes.to_string(index=False)}
```

## QAS audit
- Mean retained selected-operator energy: {full.qas_retained_selected_energy.mean():.6f}
- Mean retained support energy: {full.qas_retained_support_energy.mean():.6f}
- Mean target-reached rate: {full.qas_target_reached_rate.mean():.6f}
- Mean QAS post-warper TV: {full.qas_mean_postwarper_tv.mean():.8f}

## Interpretation boundary
Semantic automatic rates are computed only over `scoreable=True` rows. Bootstrap correlations with zero variance are exported as non-estimable, not as zero. The active-control arm is selected ex ante from probe geometry and is not a post-hoc winner.
"""
(CAMPAIGN_ROOT/"analysis_report.md").write_text(report,encoding="utf-8")

# QAS-P2 label-complete support audit.
coordinate_records=[]
for _,row in full.iterrows():
    if not str(row.variant).endswith("_qas"):
        continue
    for sketch in json.loads(row.sketches_json):
        if "mask_indices" not in sketch:
            continue
        support=set(int(i) for i in sketch["mask_indices"])
        candidates=[int(i) for i in sketch.get("baseline_candidate_indices",[])]
        candidate_set=set(candidates)
        intersection=len(support & candidate_set)
        coordinate_records.append({
            "run_key":row.run_key,"prompt_id":row.prompt_id,"seed":int(row.seed),
            "variant":row.variant,"role":row.role,"action":row.action,
            "task_type":row.task_type,"step":int(sketch["step"]),
            "vocab_size":int(sketch["vocab_size"]),
            "mask_size":len(support),
            "mask_indices":sketch["mask_indices"],
            "mask_indices_sha256":sketch["mask_indices_sha256"],
            "baseline_candidate_indices":candidates,
            "baseline_candidate_probabilities":sketch.get("baseline_candidate_probabilities",[]),
            "candidate_count":len(candidates),
            "candidate_support_intersection":intersection,
            "candidate_support_recall":intersection/max(1,len(support)),
            "candidate_support_precision":intersection/max(1,len(candidates)),
            "postwarper_tv":sketch.get("postwarper_tv"),
            "mask_jaccard":sketch.get("mask_jaccard"),
        })
coord=pd.DataFrame(coordinate_records)
coord.to_json(CAMPAIGN_ROOT/"qas_coordinate_labels.jsonl",orient="records",lines=True,force_ascii=False)
if len(coord):
    summary_coord=(coord.groupby(["variant","action","task_type"],dropna=False)
        .agg(n_steps=("step","size"),mean_mask_size=("mask_size","mean"),
             mean_candidate_support_recall=("candidate_support_recall","mean"),
             median_candidate_support_recall=("candidate_support_recall","median"),
             full_support_inside_top64_rate=("candidate_support_recall",lambda s:float((s>=1-1e-12).mean())))
        .reset_index())
else:
    summary_coord=pd.DataFrame(columns=["variant","action","task_type","n_steps","mean_mask_size","mean_candidate_support_recall","median_candidate_support_recall","full_support_inside_top64_rate"])
summary_coord.to_csv(CAMPAIGN_ROOT/"qas_candidate_pool_summary.csv",index=False)
print("QAS coordinate-label steps:",len(coord))
print(summary_coord.to_string(index=False) if len(summary_coord) else "No coordinate labels found")


# ------------------------------------------------------------------
# QAS-P2 schema-valid evidence receipts + frozen top-64 structural test
# ------------------------------------------------------------------
from jsonschema import Draft202012Validator

def _tokenizer_binding_sha256(tok):
    h = hashlib.sha256()
    backend = getattr(tok, "_tokenizer", None)
    if backend is not None and hasattr(backend, "to_str"):
        h.update(backend.to_str().encode("utf-8", errors="surrogatepass"))
    else:
        vocab = tok.get_vocab()
        for token, idx in sorted(vocab.items(), key=lambda kv: (int(kv[1]), kv[0])):
            h.update(str(int(idx)).encode("ascii"))
            h.update(b"\0")
            h.update(str(token).encode("utf-8", errors="surrogatepass"))
            h.update(b"\n")
    h.update(
        json.dumps(
            getattr(tok, "special_tokens_map", {}),
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        ).encode("utf-8")
    )
    return h.hexdigest()

TOKENIZER_HASH = _tokenizer_binding_sha256(tokenizer)
PROMPT_HASH_BY_ID = {
    spec["prompt_id"]: hashlib.sha256(spec["prompt_text"].encode("utf-8")).hexdigest()
    for spec in PROMPT_RECORDS
}

validator = Draft202012Validator(QAS_P2_RECEIPT_SCHEMA)
receipt_records = []
top64_violations = []

for _, row in full.iterrows():
    if not str(row.variant).endswith("_qas"):
        continue
    for sketch in json.loads(row.sketches_json):
        if "mask_indices" not in sketch:
            continue

        support_ids = sorted({int(i) for i in sketch["mask_indices"]})
        candidate_ids = [int(i) for i in sketch.get("baseline_candidate_indices", [])]
        candidate_probs = [float(x) for x in sketch.get("baseline_candidate_probabilities", [])]
        vocab_size = int(sketch["vocab_size"])
        support_hash = str(sketch["mask_indices_sha256"])
        prompt_hash = PROMPT_HASH_BY_ID[str(row.prompt_id)]

        fidelity_endpoint = {
            "kind": "postwarper_distribution",
            "target_reached": bool(sketch.get("target_reached", False)),
            "postwarper_tv": sketch.get("postwarper_tv"),
            "postwarper_kl": sketch.get("postwarper_kl"),
            "postwarper_jsd": sketch.get("postwarper_jsd"),
            "postwarper_support_overlap": sketch.get("postwarper_support_overlap"),
            "postwarper_topk_overlap": sketch.get("postwarper_topk_overlap"),
            "postwarper_crossings": sketch.get("postwarper_crossings"),
            "maximal_coupling_disagreement": sketch.get("maximal_coupling_disagreement"),
        }

        provenance_payload = {
            "experiment_id": EXPERIMENT_ID,
            "run_key": str(row.run_key),
            "variant": str(row.variant),
            "role": str(row.role),
            "model_revision": MODEL_REVISION_RESOLVED,
            "runtime_sha256": EXPECTED_RUNTIME_SHA256,
            "prompt_hash": prompt_hash,
            "seed": int(row.seed),
            "step_index": int(sketch["step"]),
            "operator_id": str(row.action),
            "tokenizer_hash": TOKENIZER_HASH,
            "support_hash": support_hash,
            "baseline_top64_ids": candidate_ids,
        }
        provenance_hash = hashlib.sha256(
            json.dumps(
                provenance_payload,
                sort_keys=True,
                separators=(",", ":"),
                ensure_ascii=False,
            ).encode("utf-8")
        ).hexdigest()

        receipt = {
            "schema": "qas.p2.evidence.v1",
            "run_id": str(row.run_key),
            "model_revision": str(MODEL_REVISION_RESOLVED),
            "tokenizer_hash": TOKENIZER_HASH,
            "vocab_size": vocab_size,
            "prompt_hash": prompt_hash,
            "seed": int(row.seed),
            "step_index": int(sketch["step"]),
            "operator_id": str(row.action),
            "baseline_top64_ids": candidate_ids,
            "baseline_top64_probs": candidate_probs,
            "support_ids": support_ids,
            "support_hash": support_hash,
            "support_size": len(support_ids),
            "support_fraction": len(support_ids) / max(1, vocab_size),
            "construction_parameters": dict(QAS_PARAMETERS),
            "activity_metric": sketch.get("postwarper_tv"),
            "fidelity_endpoint": fidelity_endpoint,
            "closure_test": None,
            "wall_time_ms": None,
            "gpu_time_ms": None,
            "provenance_hash": provenance_hash,
        }
        errors = sorted(validator.iter_errors(receipt), key=lambda e: list(e.path))
        if errors:
            raise RuntimeError(
                "QAS-P2 receipt schema validation failed: "
                + "; ".join(err.message for err in errors[:10])
            )
        if len(receipt["baseline_top64_ids"]) != len(receipt["baseline_top64_probs"]):
            raise RuntimeError("Top-64 ID/probability length mismatch.")
        if receipt["support_size"] != len(receipt["support_ids"]):
            raise RuntimeError("Support size mismatch.")

        outside = sorted(set(support_ids) - set(candidate_ids))
        if outside:
            top64_violations.append({
                "run_id": receipt["run_id"],
                "prompt_hash": prompt_hash,
                "seed": receipt["seed"],
                "step_index": receipt["step_index"],
                "operator_id": receipt["operator_id"],
                "support_size": receipt["support_size"],
                "outside_top64_ids": outside,
            })
        receipt_records.append(receipt)

receipt_path = CAMPAIGN_ROOT / "qas_p2_evidence_receipts.jsonl"
with receipt_path.open("w", encoding="utf-8") as h:
    for receipt in receipt_records:
        h.write(json.dumps(receipt, ensure_ascii=False, sort_keys=True) + "\n")

frozen_top64_result = {
    "schema": "qas.p2.top64_structural_test.v1",
    "experiment_id": EXPERIMENT_ID,
    "model_revision": MODEL_REVISION_RESOLVED,
    "tokenizer_hash": TOKENIZER_HASH,
    "audited_steps": len(receipt_records),
    "violation_count": len(top64_violations),
    "status": (
        "FALSIFIED"
        if top64_violations
        else "PASSED_FROZEN_CAMPAIGN"
    ),
    "claim_scope": (
        "Every prospectively audited step in this frozen campaign only; "
        "not a population-level universality claim."
    ),
    "frozen_condition": (
        "For every audited step, exact QAS support S_t is a subset of "
        "pre-operator top-64 C_64,t."
    ),
    "violations": top64_violations[:50],
}
(CAMPAIGN_ROOT / "qas_p2_universal_top64_test.json").write_text(
    json.dumps(frozen_top64_result, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

receipt_manifest = {
    "schema": "qas.p2.receipt_manifest.v1",
    "experiment_id": EXPERIMENT_ID,
    "receipt_schema": QAS_P2_RECEIPT_SCHEMA["properties"]["schema"]["const"],
    "receipt_count": len(receipt_records),
    "receipt_file": receipt_path.name,
    "receipt_file_sha256": hashlib.sha256(receipt_path.read_bytes()).hexdigest(),
    "top64_test_file": "qas_p2_universal_top64_test.json",
    "top64_test_sha256": hashlib.sha256(
        (CAMPAIGN_ROOT / "qas_p2_universal_top64_test.json").read_bytes()
    ).hexdigest(),
    "runtime_sha256": EXPECTED_RUNTIME_SHA256,
    "prompt_manifest_sha256": PROMPT_MANIFEST_SHA256,
    "model_revision": MODEL_REVISION_RESOLVED,
    "tokenizer_hash": TOKENIZER_HASH,
    "authority_boundary": "evidence_only_permission_granted_false",
}
(CAMPAIGN_ROOT / "qas_p2_receipt_manifest.json").write_text(
    json.dumps(receipt_manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("QAS-P2 evidence receipts:", len(receipt_records))
print("QAS-P2 top-64 frozen test:", frozen_top64_result["status"])
print("QAS-P2 top-64 violations:", len(top64_violations))


# Final manifest, hashes and ZIP custody.
conn.execute("PRAGMA wal_checkpoint(FULL)"); conn.commit(); export_current_tables()
output_names=[p.name for p in CAMPAIGN_ROOT.iterdir() if p.is_file() and not p.name.endswith('.zip')]
manifest=[]
for name in sorted(output_names):
    p=CAMPAIGN_ROOT/name; manifest.append({"path":name,"bytes":p.stat().st_size,"sha256":hashlib.sha256(p.read_bytes()).hexdigest()})
(CAMPAIGN_ROOT/"run_manifest.json").write_text(json.dumps({"experiment_id":EXPERIMENT_ID,"software_version":SOFTWARE_VERSION,"files":manifest},indent=2),encoding="utf-8")
(CAMPAIGN_ROOT/"SHA256SUMS.txt").write_text("\n".join(f"{x['sha256']}  {x['path']}" for x in manifest)+"\n",encoding="utf-8")
final_zip=WORK_ROOT/f"{EXPERIMENT_ID}_FINAL_COMPLETE.zip"; partial=final_zip.with_suffix('.zip.partial')
if partial.exists(): partial.unlink()
with zipfile.ZipFile(partial,"w",zipfile.ZIP_DEFLATED,allowZip64=True) as z:
    for p in sorted(CAMPAIGN_ROOT.iterdir()):
        if p.is_file(): z.write(p,p.name)
with zipfile.ZipFile(partial) as z: assert z.testzip() is None
os.replace(partial,final_zip)
print("✅ FINAL",final_zip,"SHA256",hashlib.sha256(final_zip.read_bytes()).hexdigest())
print(report)
